# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def print_feature_as_dict(features_dict):
    print("{")
    for k, v in features_dict.items():
        print(f"    '{k}': {v.__name__},")  # int, float, str 등
    print("}")


## Initial Setup

- Source to Consider

# Data

In [6]:
import yaml

# YAML 파일 경로
yaml_path = "../config/feature_kn.yaml"

# YAML 파일 열고 읽기
with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

# 내용 출력
print(config.keys())

dict_keys(['7DT_20', '7DT_40', 'Rubin+7DT_20', 'Rubin+7DT_40'])


In [15]:
lsst_filters = ["u", "g", "r", "i", "z", "y"]

In [19]:
config

{'7DT_20': {'color': ['m400-m425',
   'm400-m450',
   'm400-m475',
   'm400-m500',
   'm400-m525',
   'm400-m550',
   'm400-m575',
   'm400-m600',
   'm400-m625',
   'm400-m650',
   'm400-m675',
   'm400-m700',
   'm400-m725',
   'm400-m750',
   'm400-m775',
   'm400-m800',
   'm400-m825',
   'm400-m850',
   'm400-m875',
   'm425-m450',
   'm425-m475',
   'm425-m500',
   'm425-m525',
   'm425-m550',
   'm425-m575',
   'm425-m600',
   'm425-m625',
   'm425-m650',
   'm425-m675',
   'm425-m700',
   'm425-m725',
   'm425-m750',
   'm425-m775',
   'm425-m800',
   'm425-m825',
   'm425-m850',
   'm425-m875',
   'm450-m475',
   'm450-m500',
   'm450-m525',
   'm450-m550',
   'm450-m575',
   'm450-m600',
   'm450-m625',
   'm450-m650',
   'm450-m675',
   'm450-m700',
   'm450-m725',
   'm450-m750',
   'm450-m775',
   'm450-m800',
   'm450-m825',
   'm450-m850',
   'm450-m875',
   'm475-m500',
   'm475-m525',
   'm475-m550',
   'm475-m575',
   'm475-m600',
   'm475-m625',
   'm475-m650',
   'm

## 20 bands

In [18]:
# ugrizy를 key로 갖는 dict 정의
features_broadband = {f"{band}20": [] for band in lsst_filters}
features_med20 = []

# 색상 feature 구분
for feature in config['Rubin+7DT_20']['color']:
    if len(feature) == 9:  # medium-band color (e.g. m400-m425)
        features_med20.append(feature)
    else:
        found = False
        for band in features_broadband:
            if (f"-{band[0]}" in feature) | (f"{band[0]}-" in feature):
                features_broadband[band].append(feature)
                print(feature)
                found = True
                break
        if not found:
            print("Unknown feature:", feature)


m400-u
m425-u
m450-u
m475-u
m500-u
m525-u
m550-u
m575-u
m600-u
m625-u
m650-u
m675-u
m700-u
m725-u
m750-u
m775-u
m800-u
m825-u
m850-u
m875-u
m400-g
m425-g
m450-g
m475-g
m500-g
m525-g
m550-g
m575-g
m600-g
m625-g
m650-g
m675-g
m700-g
m725-g
m750-g
m775-g
m800-g
m825-g
m850-g
m875-g
m400-r
m425-r
m450-r
m475-r
m500-r
m525-r
m550-r
m575-r
m600-r
m625-r
m650-r
m675-r
m700-r
m725-r
m750-r
m775-r
m800-r
m825-r
m850-r
m875-r
m400-i
m425-i
m450-i
m475-i
m500-i
m525-i
m550-i
m575-i
m600-i
m625-i
m650-i
m675-i
m700-i
m725-i
m750-i
m775-i
m800-i
m825-i
m850-i
m875-i
m400-z
m425-z
m450-z
m475-z
m500-z
m525-z
m550-z
m575-z
m600-z
m625-z
m650-z
m675-z
m700-z
m725-z
m750-z
m775-z
m800-z
m825-z
m850-z
m875-z
m400-y
m425-y
m450-y
m475-y
m500-y
m525-y
m550-y
m575-y
m600-y
m625-y
m650-y
m675-y
m700-y
m725-y
m750-y
m775-y
m800-y
m825-y
m850-y
m875-y


In [17]:
filte = "g"

band_features = features_broadband[f"{filte}20"]

features_dict = {
    'Sample_ID': int,
    'Class': str,
    'uid': str
}

for feature in features_med20+band_features:
    features_dict[feature] = float

print_feature_as_dict(features_dict)

{
    'Sample_ID': int,
    'Class': str,
    'uid': str,
    'm400-m425': float,
    'm400-m450': float,
    'm400-m475': float,
    'm400-m500': float,
    'm400-m525': float,
    'm400-m550': float,
    'm400-m575': float,
    'm400-m600': float,
    'm400-m625': float,
    'm400-m650': float,
    'm400-m675': float,
    'm400-m700': float,
    'm400-m725': float,
    'm400-m750': float,
    'm400-m775': float,
    'm400-m800': float,
    'm400-m825': float,
    'm400-m850': float,
    'm400-m875': float,
    'm425-m450': float,
    'm425-m475': float,
    'm425-m500': float,
    'm425-m525': float,
    'm425-m550': float,
    'm425-m575': float,
    'm425-m600': float,
    'm425-m625': float,
    'm425-m650': float,
    'm425-m675': float,
    'm425-m700': float,
    'm425-m725': float,
    'm425-m750': float,
    'm425-m775': float,
    'm425-m800': float,
    'm425-m825': float,
    'm425-m850': float,
    'm425-m875': float,
    'm450-m475': float,
    'm450-m500': float,
    'm